In [1]:

!pip install -q -U youtube-transcript-api langchain langchain-community \
   langchain-google-genai faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 101.1 MB/s eta 0:00:00


In [2]:
# =========================================================
# AskTheVideo — YouTube Video Q&A Chatbot (RAG-based)
# Updated for 2026: Gemini (free tier) + current library APIs
# =========================================================

# ---- Step 0: Install dependencies ----
# !pip install -q -U youtube-transcript-api langchain langchain-community \
#                    langchain-google-genai faiss-cpu tiktoken python-dotenv

import os
import getpass

In [3]:
# ---- Step 0a: Set your Gemini API key ----
# Get a free key at https://aistudio.google.com/apikey
# NEVER hardcode it in a notebook you might share — use getpass or Colab secrets instead.
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


In [5]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_9687/4069737906.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:


# =========================================================
# Step 1a — Indexing: Document Ingestion (transcript fetch)
# =========================================================
def get_transcript(video_id: str, languages=("en",)) -> str:
    """
    Fetches a YouTube transcript using the current (2026) youtube-transcript-api.
    Note: the old YouTubeTranscriptApi.get_transcript(...) static method was
    removed. You now instantiate the class and call .fetch().
    """
    ytt_api = YouTubeTranscriptApi()
    try:
        fetched = ytt_api.fetch(video_id, languages=list(languages))
        # fetched is a FetchedTranscript object; each item has .text, .start, .duration
        transcript = " ".join(snippet.text for snippet in fetched)
        return transcript
    except TranscriptsDisabled:
        raise RuntimeError("Captions are disabled for this video.")
    except NoTranscriptFound:
        raise RuntimeError(f"No transcript found in languages: {languages}")
    except VideoUnavailable:
        raise RuntimeError("Video is unavailable (private, deleted, or region-locked).")
    except Exception as e:
        # This is the common failure mode on Colab: YouTube blocks
        # requests coming from Google Cloud / datacenter IP ranges.
        # If you hit this repeatedly, you'll need a residential proxy
        # (the library supports WebshareProxyConfig — see its README)
        # or a hosted transcript API (e.g. Supadata) as a fallback.
        raise RuntimeError(
            f"Transcript fetch failed ({e}). If this happens consistently on "
            "Colab, it's likely YouTube blocking the datacenter IP — "
            "consider a proxy or a hosted transcript API as fallback."
        )


video_id = "Gfr50f6ZBvo"  # only the ID, not the full URL
transcript = get_transcript(video_id)
print(transcript[:500], "...")


the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful  ...


In [8]:
# =========================================================
# Step 1b — Indexing: Text Splitting
# =========================================================
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print(f"Number of chunks: {len(chunks)}")



Number of chunks: 168


In [11]:
import time
from google.genai.errors import ClientError

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

BATCH_SIZE = 20          # chunks per request batch — stay well under the 100/min cap
PAUSE_SECONDS = 15       # pause between batches
MAX_RETRIES = 5

def embed_in_batches(docs, embeddings_model, batch_size=BATCH_SIZE, pause=PAUSE_SECONDS):
    vector_store = None
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i + batch_size]
        attempt = 0
        while True:
            try:
                if vector_store is None:
                    vector_store = FAISS.from_documents(batch, embeddings_model)
                else:
                    vector_store.add_documents(batch)
                break
            except ClientError as e:
                attempt += 1
                if attempt > MAX_RETRIES or "RESOURCE_EXHAUSTED" not in str(e):
                    raise
                wait = pause * attempt
                print(f"Rate limited on batch {i // batch_size + 1}, retrying in {wait}s...")
                time.sleep(wait)
        print(f"Embedded batch {i // batch_size + 1}/{(len(docs) - 1) // batch_size + 1} ({len(batch)} chunks)")
        time.sleep(pause)
    return vector_store

vector_store = embed_in_batches(chunks, embeddings)

Embedded batch 1/9 (20 chunks)
Embedded batch 2/9 (20 chunks)
Embedded batch 3/9 (20 chunks)
Embedded batch 4/9 (20 chunks)
Embedded batch 5/9 (20 chunks)
Embedded batch 6/9 (20 chunks)
Embedded batch 7/9 (20 chunks)
Embedded batch 8/9 (20 chunks)
Embedded batch 9/9 (8 chunks)


In [12]:
# =========================================================
# Step 2 — Retrieval
# =========================================================
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})



In [13]:



# =========================================================
# Step 3 — Augmentation
# =========================================================
# gemini-2.5-flash: fast, cheap, generous free tier — good default for this use case.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

prompt = PromptTemplate(
    template="""
      You are a helpful assistant that answers questions about a YouTube video's
      transcript.
      Answer ONLY from the provided transcript context.
      If the context is insufficient to answer, say you don't know — do not
      make anything up.

      Transcript context:
      {context}

      Question: {question}
    """,
    input_variables=["context", "question"],
)


In [14]:


# =========================================================
# Step 4 — Chain (LCEL) — same pattern you had, still current best practice
# =========================================================
def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough(),
})

parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser



In [20]:
# =========================================================
# Try it out
# =========================================================
question = "How to make Cakes"
answer = main_chain.invoke(question)
print(answer)

I don't know. The provided transcript context does not contain information on how to make cakes.
